In [1]:
import os
import math
import json
import copy
import itertools
import functools
from datetime import datetime

In [3]:
import torch
import numpy as np 
from torch import nn
import matplotlib.pyplot as plt
from rich.progress import Progress

from fintorch.deep.model import *
from fintorch.deep.module import create_module_from_args
from fintorch.deep.transform.feature import *
from fintorch.deep.transform.label import *
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.enum import TimeFrame
from fintorch.deep.metrics import Metrics
from fintorch.setting import RICH_PROGRESS_COLUMNS
from fintorch.utils.timestamp import create_timestamps
from fintorch.utils.args import DefaultArgumentParser

In [30]:
string_args = [
    "--start-date", "2020-06-01",
    "--stop-date", "2024-01-01",
    
    "--num-hidden-layers", "2",
    
    "--shuffle", "true",
    "--val-length", "1000",
    "--test-length", "1000",
    "--train-length", "10000",

    "--batch-size", "128",
    "--epochs-count", "10",
    
    "--dropout", "0.5",
    "--weight-decay", "0.1",
]

args = DefaultArgumentParser.parse(args=string_args)

In [31]:
module = create_module_from_args(
    feature_transform_type=PreviousFractalsFeatureTransform,
    label_transform_type=ForwardMiddleSmaLabelTransform,
    model_type=LSTM,
    args=args
)

In [32]:
dc = ONLINE_EXCHANGE.future.data.get_data_collection(symbols=[args.symbol], time_frames=[args.time_frame])
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()
sdf.insert(0, "datetime", [datetime.fromtimestamp(ts) for ts in sdf.index.to_list()])

sdf

,datetime,open,high,low,close,volume,trade
timestamp,,,,,,,
1569456000,2019-09-26 03:30:00,8428.15,8464.40,7700.67,8060.15,60065.945,449.0
1569542400,2019-09-27 03:30:00,8061.98,8260.00,7851.94,8176.52,40034.653,504.0
1569628800,2019-09-28 03:30:00,8178.26,8311.00,8001.14,8195.38,33820.919,756.0
1569715200,2019-09-29 03:30:00,8199.00,8226.35,7888.98,8041.96,35966.056,1138.0
1569801600,2019-09-30 03:30:00,8042.08,8331.89,7709.01,8285.84,54412.520,751.0
...,...,...,...,...,...,...,...
1710115200,2024-03-11 03:30:00,69001.80,72890.00,67000.00,72153.80,465667.449,33684.0
1710201600,2024-03-12 03:30:00,72153.90,73370.00,68600.00,71503.20,407956.906,37957.0
1710288000,2024-03-13 03:30:00,71503.30,73755.00,71388.10,73127.60,288446.338,43766.0


In [60]:
LENGTH = 1

# indicators
df = sdf.copy()
df["tr"] = df.high / df.low - 1
df["roc"] = df.close / df.open - 1
df["atr"] = df.tr.rolling(window=LENGTH).mean()
df["aroc"] = np.abs(df.roc).rolling(window=LENGTH).mean()

with module:
    y_hats_dict = module.predict(dc=dc, timestamps=df.index.to_list(), mode="val")
    
    prediction_dict = {k: np.argmax(v) for k, v in y_hats_dict.items() if v is not None}
    prediction_dict.update({k: np.nan for k, v in y_hats_dict.items() if v is None})
    df["prediction"] = prediction_dict.values()
    df["buy"] = 1 == df.prediction

df = df.dropna()

df

,datetime,open,high,low,close,volume,trade,tr,roc,atr,aroc,prediction,buy
timestamp,,,,,,,,,,,,,
1569456000,2019-09-26 03:30:00,8428.15,8464.40,7700.67,8060.15,60065.945,449.0,0.099177,-0.043663,0.099177,0.043663,1.0,True
1569542400,2019-09-27 03:30:00,8061.98,8260.00,7851.94,8176.52,40034.653,504.0,0.051969,0.014207,0.051969,0.014207,1.0,True
1569628800,2019-09-28 03:30:00,8178.26,8311.00,8001.14,8195.38,33820.919,756.0,0.038727,0.002093,0.038727,0.002093,1.0,True
1569715200,2019-09-29 03:30:00,8199.00,8226.35,7888.98,8041.96,35966.056,1138.0,0.042765,-0.019154,0.042765,0.019154,1.0,True
1569801600,2019-09-30 03:30:00,8042.08,8331.89,7709.01,8285.84,54412.520,751.0,0.080799,0.030311,0.080799,0.030311,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1669766400,2022-11-30 03:30:00,16433.30,17248.30,16421.50,17153.30,620324.950,17037.0,0.050349,0.043813,0.050349,0.043813,1.0,True
1669852800,2022-12-01 03:30:00,17153.40,17348.50,16851.20,16970.50,411051.309,36453.0,0.029511,-0.010663,0.029511,0.010663,1.0,True
1669939200,2022-12-02 03:30:00,16970.50,17096.30,16760.00,17083.00,335039.215,14145.0,0.020066,0.006629,0.020066,0.006629,1.0,True


In [81]:
TPR = 12
SLR = 4

tdf = df.copy()

#######################
###### backtest #######
#######################
profits_list, bars_list = [], []

entry_price = None
for index in tqdm(range(len(tdf))):
    if entry_price is None and tdf.buy.iloc[index]:
        entry_price = tdf.close.iloc[index]
        stop_loss = tdf.aroc.iloc[index] * SLR
        take_profit = tdf.aroc.iloc[index] * TPR
    
        stop_loss_price = round(entry_price * (1 - stop_loss - 0.0002 * 2), 2)
        take_profit_price = round(entry_price * (1 + take_profit + 0.0002 * 2), 2)

        # print(stop_loss_price)
        # print(take_profit_price)
        
    elif entry_price is not None:
        profit = None
        if tdf.low.iloc[index] <= stop_loss_price:
            profit = -stop_loss
        elif take_profit_price <= tdf.high.iloc[index]:
            profit = take_profit

        if profit is not None:
            profits_list.append(profit)
            entry_price = None

100%|██████████| 1166/1166 [00:00<00:00, 55483.74it/s]


In [84]:
LEVERAGE = 50

profits = np.array(profits_list) * LEVERAGE

days = len(tdf) * TimeFrame.DAY1 // TimeFrame.DAY1

trades = len(profits)
daily_trades = len(profits) // days

net_profit = profits.sum()
compound_profit = (1 + profits).prod() - 1
daily_net_profit = net_profit / days
daily_compound_profit = (compound_profit + 1) ** (1 / days) - 1

wining_ratio = (0 < profits).sum() / len(profits) * 100

# print statements
print("{:<32}{}".format("Days", days))
print()
print("{:<32}{}".format("Trades", trades))
print("{:<32}{}".format("Daily trades", daily_trades))
print()
print("{:<32}{:.2f}".format("Net profit", net_profit))
print("{:<32}{:.2f}".format("Compound profit", compound_profit))
print("{:<32}{:.2f}".format("Daily net profit", daily_net_profit))
print("{:<32}{:.2f}".format("Daily compound profit", daily_compound_profit))
print()
print("{:<32}{:.2f}%".format("Wining ratio", wining_ratio))


Days                            1166

Trades                          30
Daily trades                    0

Net profit                      69.17
Compound profit                 625994748828.00
Daily net profit                0.06
Daily compound profit           0.02

Wining ratio                    33.33%
